# Field strengths

`FS(group, mu, nu)` is abelian and takes no adjoint index.
`FS(group, mu, nu, a)` is non-abelian and requires the adjoint index `a`.
A product of $n$ non-abelian field strengths expands into $3^n$ local monomials.


## Setup


In [1]:
import re
import sys
from fractions import Fraction
from pathlib import Path

from symbolica import Expression, S

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None, simplify=True):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = (
            sum(source_terms[1:], source_terms[0])
            if len(source_terms) > 1
            else source_terms[0]
        )
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False, simplify=simplify))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False, simplify=simplify))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)

from feynpy import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    Field,
    FS,
    Gamma,
    GaugeGroup,
    GaugeRepresentation,
    Model,
    StructureConstant,
    T,
    dirac_field,
)
from symbolic.spenso_structures import (
    lorentz_levi_civita,
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)


In [2]:
mu, nu, rho, sigma = S("mu"), S("nu"), S("rho"), S("sigma")
a, b, c = S("a"), S("b"), S("c")
aC, aW = S("aC"), S("aW")
eQED, gS, g2 = S("eQED"), S("gS"), S("g2")

Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Gluon = Field("G", spin=1, self_conjugate=True, symbol=S("G0"), indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
W = Field("W", spin=1, self_conjugate=True, symbol=S("W0"), indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
Psi = dirac_field("Psi", symbol=S("psi"), conjugate_symbol=S("psibar"))

U1QED = GaugeGroup(name="U1QED", abelian=True, coupling=eQED, gauge_boson=Photon, charge="Q")
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=Gluon,
    structure_constant=structure_constant,
    representations=(
        GaugeRepresentation(index=COLOR_FUND_INDEX, generator_builder=gauge_generator, name="fund"),
    ),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    structure_constant=weak_structure_constant,
    representations=(
        GaugeRepresentation(index=WEAK_FUND_INDEX, generator_builder=weak_gauge_generator, name="doublet"),
    ),
)


## Kinetic terms


In [3]:
photon_kinetic = Model(
    -(Expression.num(1) / Expression.num(4)) * FS(U1QED, mu, nu) * FS(U1QED, mu, nu)
)
show_model(photon_kinetic)

gluon_kinetic = Model(
    -(Expression.num(1) / Expression.num(4)) * FS(SU3C, mu, nu, aC) * FS(SU3C, mu, nu, aC)
)
show_model(gluon_kinetic)


Lagrangian
-1/4 * FS(U1QED, mu, nu) * FS(U1QED, mu, nu)

Feynman Rules
1 vertex signature(s)

Vertex: ('A', 'A')
Rule: -1𝑖*pcomp(q1,mu2)*pcomp(q2,mu1)+1𝑖*g(mink(4, mu1),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)

Lagrangian
-1/4 * FS(SU3C, mu, nu, aC) * FS(SU3C, mu, nu, aC)

Feynman Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu2)*pcomp(q2,mu1)

Vertex: ('G', 'G', 'G')
Rule: gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu3)-gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu3)-gS*g(mink(4, mu1),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)+gS*g(mink(4, mu1),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q3,mu2)+gS*g(mink(4, mu2),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu1)-gS*g(mink(4, mu2),mink(4, mu3))*f(coad(8, a1),

## Higher operators

$\varepsilon F F$, $f^{abc} F^a F^b F^c$, and $F^4$ expand into many local
terms. For those operators we print the Lagrangian and the compiled term
count rather than every high-leg vertex.


In [4]:
epsFF = Model(
    S("theta") * lorentz_levi_civita(mu, nu, rho, sigma) * FS(U1QED, mu, nu) * FS(U1QED, rho, sigma)
)
show_model(epsFF)

f3 = Model(
    S("c3")
    * StructureConstant(a, b, c)
    * FS(SU3C, mu, nu, a)
    * FS(SU3C, nu, rho, b)
    * FS(SU3C, rho, mu, c)
)
show("Lagrangian", f3.lagrangian_decl.source_terms[0])
show("Compiled local terms", len(f3.lagrangian().terms))

f4 = Model(
    S("lam")
    * FS(SU3C, mu, nu, a) * FS(SU3C, mu, nu, a)
    * FS(SU3C, rho, sigma, b) * FS(SU3C, rho, sigma, b)
)
show("Lagrangian", f4.lagrangian_decl.source_terms[0])
show("Compiled local terms", len(f4.lagrangian().terms))

mixed = Model(
    S("kappa")
    * FS(SU3C, mu, nu, aC) * FS(SU3C, mu, nu, aC)
    * FS(SU2L, rho, sigma, aW) * FS(SU2L, rho, sigma, aW),
    gauge_groups=(SU3C, SU2L),
    fields=(Gluon, W),
)
show("Lagrangian", mixed.lagrangian_decl.source_terms[0])
show("Compiled local terms", len(mixed.lagrangian().terms))


Lagrangian
theta*lor_levi_civita(mink(4, mu),mink(4, nu),mink(4, rho),mink(4, sigma)) * FS(U1QED, mu, nu) * FS(U1QED, rho, sigma)

Feynman Rules
1 vertex signature(s)

Vertex: ('A', 'A')
Rule: -1𝑖*theta*lor_levi_civita(mink(4, mu1),mink(4, mu1_int),mink(4, mu2),mink(4, mu2_int))*pcomp(q1,mu1_int)*pcomp(q2,mu2_int)+1𝑖*theta*lor_levi_civita(mink(4, mu1),mink(4, mu1_int),mink(4, mu2_int),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu2_int)-1𝑖*theta*lor_levi_civita(mink(4, mu2),mink(4, mu1_int),mink(4, mu1),mink(4, mu2_int))*pcomp(q1,mu2_int)*pcomp(q2,mu1_int)+1𝑖*theta*lor_levi_civita(mink(4, mu2),mink(4, mu1_int),mink(4, mu2_int),mink(4, mu1))*pcomp(q1,mu2_int)*pcomp(q2,mu1_int)+1𝑖*theta*lor_levi_civita(mink(4, mu1_int),mink(4, mu1),mink(4, mu2),mink(4, mu2_int))*pcomp(q1,mu1_int)*pcomp(q2,mu2_int)-1𝑖*theta*lor_levi_civita(mink(4, mu1_int),mink(4, mu1),mink(4, mu2_int),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu2_int)+1𝑖*theta*lor_levi_civita(mink(4, mu1_int),mink(4, mu2),mink(4, mu1),mink(4

## Field strength coupled to fermions

Positional labels follow `field.indices`. For a colored fermion,
`PsiColor.bar(s1, i)` is spinor `s1` and color `i`.


In [5]:
s1, s2, s3, i, j = S("s1"), S("s2"), S("s3"), S("i"), S("j")

fs_psi = Model(
    FS(U1QED, mu, nu) * Psi.bar(s1) * Gamma(s1, s2, mu) * Gamma(s2, s3, nu) * Psi(s3)
)
show_model(fs_psi, Psi.bar, Psi, Photon)

PsiColor = Field(
    "PsiC",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psiC"),
    conjugate_symbol=S("psibarC"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
)
fs_quark = Model(
    FS(SU3C, mu, nu, a)
    * PsiColor.bar(s1, i)
    * Gamma(s1, s2, mu)
    * Gamma(s2, s3, nu)
    * T(a, i, j)
    * PsiColor(s3, j)
)
show_model(fs_quark, PsiColor.bar, PsiColor, Gluon)


Lagrangian
gamma(bis(4, s1),bis(4, s2),mink(4, mu))*gamma(bis(4, s2),bis(4, s3),mink(4, nu)) * FS(U1QED, mu, nu) * Psi.bar * Psi

Feynman Rule
-gamma(bis(4, s2),bis(4, i2),mink(4, mu1_int))*gamma(bis(4, i1),bis(4, s2),mink(4, mu3))*pcomp(q3,mu1_int)+gamma(bis(4, s2),bis(4, i2),mink(4, mu3))*gamma(bis(4, i1),bis(4, s2),mink(4, mu1_int))*pcomp(q3,mu1_int)

Lagrangian
gamma(bis(4, s1),bis(4, s2),mink(4, mu))*gamma(bis(4, s2),bis(4, s3),mink(4, nu))*t(coad(8, a),cof(3, i),cof(3, j)) * FS(SU3C, mu, nu, a) * PsiC.bar * PsiC

Feynman Rule
-gamma(bis(4, s2),bis(4, i2),mink(4, mu1_int))*gamma(bis(4, i1),bis(4, s2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(q3,mu1_int)+gamma(bis(4, s2),bis(4, i2),mink(4, mu3))*gamma(bis(4, i1),bis(4, s2),mink(4, mu1_int))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(q3,mu1_int)



## Declaration errors

A non-abelian field strength must carry an adjoint index. An abelian one
must not. Every adjoint index must be contracted.


In [6]:
try:
    Model(-(Expression.num(1) / Expression.num(4)) * FS(SU3C, mu, nu) * FS(SU3C, mu, nu)).lagrangian()
except ValueError as exc:
    show("non-abelian FS without adjoint index", exc)

try:
    Model(-(Expression.num(1) / Expression.num(4)) * FS(U1QED, mu, nu, a) * FS(U1QED, mu, nu, a)).lagrangian()
except ValueError as exc:
    show("abelian FS with adjoint index", exc)

try:
    Model(S("c") * FS(SU3C, mu, nu, a) * FS(SU3C, rho, sigma, b)).lagrangian()
except ValueError as exc:
    show("open adjoint index", exc)


non-abelian FS without adjoint index
FieldStrength compilation: non-abelian gauge group 'SU3C' field strength requires an explicit adjoint index, e.g. FieldStrength(group, mu, nu, a).

abelian FS with adjoint index
FieldStrength compilation: abelian gauge group 'U1QED' field strength does not take an adjoint index; write FieldStrength(group, mu, nu).

open adjoint index
FieldStrength monomial has open (uncontracted) adjoint index/indices ['python::{}::a', 'python::{}::b']; the Lagrangian term must be a gauge singlet. Contract them against another field strength or a tensor carrying the matching adjoint label(s).

